In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os

In [6]:
TEST_CSV = r"C:\Users\cocod\Work\Repos\NFL_stats_tracker\stats\season_datasets\raw_combined_datasets\test_dataset.csv"
VALIDATE_CSV = r"C:\Users\cocod\Work\Repos\NFL_stats_tracker\stats\season_datasets\raw_combined_datasets\validate_dataset.csv"
TRAIN_CSV = r"C:\Users\cocod\Work\Repos\NFL_stats_tracker\stats\season_datasets\raw_combined_datasets\train_dataset.csv"

In [24]:
def pull_train_validate_test_datasets(train_csv:str, validate_csv:str, test_csv:str):
    if not os.path.exists(train_csv):
        print(f'ERROR: the train dataset does not exist, {train_csv}\n')
    
    if not os.path.exists(validate_csv):
        print(f'ERROR: the validate dataset does not exist, {validate_csv}\n')
    
    if not os.path.exists(test_csv):
        print(f'ERROR: the test dataset does not exist, {test_csv}\n')
    
    df_train = pd.read_csv(train_csv)
    df_validate = pd.read_csv(validate_csv)
    df_test = pd.read_csv(test_csv)
    
    return df_train, df_validate, df_test

In [38]:
import sys
NFL_STATS_DIR = r'..\\'
sys.path.append(NFL_STATS_DIR)
import nfl_stats_scraper_constants as nssc

def remove_special_cols(df_train:pd.DataFrame, df_validate:pd.DataFrame, df_test:pd.DataFrame):
    cols_set = df_train.columns.to_list().copy()
    print(f'INFO: col set start size, {len(cols_set)}')
    
    # remove the kicking
    kicking_cols = [val[0][3:] for val in nssc.KICKING_COLUMN_MAP.values()]
    tgt_cols = list()
    for col in cols_set:
        found_match = False
        for c in kicking_cols:
            if c in col:
                found_match = True
                break
        if not found_match:
            tgt_cols.append(col)
    print(f'INFO: col set after removed kicking, {len(tgt_cols)}')
    
    # remove returns
    tgt2_cols = list()
    returns_cols = [val[0][3:] for val in nssc.RETURNS_COLUMN_MAP.values()]
    for col in tgt_cols:
        found_match = False
        for c in returns_cols:
            if c in col:
                found_match = True
                break
        if not found_match:
            tgt2_cols.append(col)
    print(f'INFO: col set after removed returns, {len(tgt2_cols)}')
    
    # remove last 5 games
    s = 'last 5 games'
    tgt3_cols = [col for col in tgt2_cols if not(s in col)]
    print(f'INFO: col set after removed last 5 games, {len(tgt3_cols)}')
    
    # remove Second Quarter
    s2nd = 'Second Quarter Pts'
    tgt4_cols = [col for col in tgt3_cols if not(s2nd in col)]
    print(f'INFO: col set after removed {s2nd}, {len(tgt4_cols)}')
    
    # remove Third Quarter
    s3rd = 'Third Quarter'
    tgt5_cols = [col for col in tgt4_cols if not(s3rd in col)]
    print(f'INFO: col set after removed {s3rd}, {len(tgt5_cols)}')
    
    # remove player offense set
    cols_po = [val[0][3:] for val in nssc.PLAYER_OFFENSE_COLUMN_MAP.values()]
    cols_po

    po_cols_keep = ['Passing Long', 'Passing Rate', 'Rushing Long',  'Offense Fumbles', 'Offense Fumbles Yards Loss']
    
    for col in po_cols_keep:
        cols_po.remove(col)
    
    # remove redundant player offense columns/features
    tgt6_cols = list()

    for col in tgt5_cols:
        found_match = False
        for c in cols_po:
            if c in col:
                found_match = True
                break
        if not found_match:
            tgt6_cols.append(col)
    print(f'INFO: col set after removed player offense, {len(tgt6_cols)}')
    
    df_train_o = df_train[tgt6_cols]
    df_validate_o = df_validate[tgt6_cols]
    df_test_o = df_test[tgt6_cols]
    
    print(f'INFO: Length of df train {df_train_o.shape}')
    print(f'INFO: Length of df validate {df_validate_o.shape}')
    print(f'INFO: Length of df test {df_test_o.shape}')
    
    return df_train_o, df_validate_o, df_test_o

In [8]:
def drop_unwanted_columns(df_train:pd.DataFrame, df_validate:pd.DataFrame, df_test:pd.DataFrame, cols_remove:list):
    cols_all = df_train.columns.to_list().copy()
    tgt_cols = [col for col in cols_all if not(call in cols_remove)]
    df_tgt_train = df_train[tgt_cols]
    df_tgt_validate = df_validate[tgt_cols]
    df_tgt_test = df_test[tgt_cols]
    
    return df_tgt_train, df_tgt_validate, df_tgt_test

In [50]:
def replace_any_nan_values(df_train:pd.DataFrame, df_validate:pd.DataFrame, df_test:pd.DataFrame):
    df_train_o = df_train.copy()
    for col in df_train.columns:
        lorig = len(df_train[col])
        ldrop = len(df_train[col].dropna())
        if lorig != ldrop:
            print(f'Found a difference for train {col}, {lorig} vs {ldrop}')
            df_train_o[col] = df_train[col].fillna(0)
    
    df_validate_o = df_validate.copy()
    for col in df_validate.columns:
        lorig = len(df_validate[col])
        ldrop = len(df_validate[col].dropna())
        if lorig != ldrop:
            print(f'Found a difference for validate {col}, {lorig} vs {ldrop}')
            df_validate_o[col] = df_validate[col].fillna(0)
    
    df_test_o = df_test.copy()
    for col in df_test.columns:
        lorig = len(df_test[col])
        ldrop = len(df_test[col].dropna())
        if lorig != ldrop:
            print(f'Found a difference for test {col}, {lorig} vs {ldrop}')
            df_test_o[col] = df_test[col].fillna(0)
    
    return df_train_o, df_validate_o, df_test_o

In [54]:
def get_feature_labels_train_validate_test_sets(df_train:pd.DataFrame, df_validate:pd.DataFrame, df_test:pd.DataFrame):
    train_cols = df_train.columns.to_list().copy()
    
    takeout_cols = ['Year', 'Week', 'Home Team', 'Away Team']
    tgt_cols = [col for col in train_cols if not(col in takeout_cols)]
    
    x_cols = [col for col in tgt_cols if 'Winning Team' != col]
    
    y_col = ['Winning Team'] # 1 for home team and 0 for away team
    
    x_train = df_train[x_cols].to_numpy()
    print(f'INFO: The shape of the training feature set is {x_train.shape}')
    y_train = df_train[y_col].to_numpy().ravel()
    print(f'INFO: The shape of the training labels is {y_train.shape}')
    
    x_validate = df_validate[x_cols].to_numpy()
    print(f'INFO: The shape of the validate feature set is {x_validate.shape}')
    y_validate = df_validate[y_col].to_numpy().ravel()
    print(f'INFO: The shape of the validate labels is {y_validate.shape}')
    
    x_test = df_test[x_cols].to_numpy()
    print(f'INFO: The shape of the test feature set is {x_test.shape}')
    y_test = df_test[y_col].to_numpy().ravel()
    print(f'INFO: The shape of the test labels is {y_test.shape}')
    
    return x_train, y_train, x_validate, y_validate, x_test, y_test

In [80]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

def normalize_test_validate_train_sets(x_train:np.ndarray, x_test:np.ndarray, x_validate:np.ndarray):
    scaler = StandardScaler()
    scaler.fit(x_train)
    x_train_norm = scaler.transform(x_train)
    x_validate_norm = scaler.transform(x_validate)
    x_test_norm = scaler.transform(x_test)
    
    return x_train_norm, x_validate_norm, x_test_norm

In [71]:
from sklearn import svm
from sklearn import metrics
def run_svm_nfl_predict(iX_train:np.ndarray, iY_train:np.ndarray,
                        iX_validate:np.ndarray, iY_validate:np.ndarray,
                        iX_test:np.ndarray, iY_test:np.ndarray,
                        kernel_type:str='rbf', C_in:float=1.0, degree_in:int=3):
    # Expects the data to be normalized
    
    try:
        clf = svm.SVC(kernel=kernel_type, cache_size=500, C=C_in, degree=degree_in)
        #clf = svm.SVC(kernel=kernel_type, cache_size=500, probability=True)
    except Exception as e:
        print(f"ERROR: failed to create an svm object for kernel {kernel_type}. Reason -> {e}\n")
        return
    
    # train the classifier
    clf.fit(iX_train, iY_train)
    
    # Run against the validation set
    y_pred_validate = clf.predict(iX_validate)
    print(f'Validate Set Accuracy: {metrics.accuracy_score(iY_validate, y_pred_validate)}')
    
    # Run against the test set
    y_pred_test = clf.predict(iX_test)
    print(f'Test Set Accuracy: {metrics.accuracy_score(iY_test, y_pred_test)}')
    
    return clf

In [5]:
def run_mlp():
    pass

In [43]:
#
# Pull CSV Data
#
df_train, df_validate, df_test = pull_train_validate_test_datasets(TRAIN_CSV, VALIDATE_CSV, TEST_CSV)

In [51]:
#
# Drop Unwanted Features/Columns
#
df_train_o, df_validate_o, df_test_o = remove_special_cols(df_train, df_validate, df_test)

INFO: col set start size, 932
INFO: col set after removed kicking, 852
INFO: col set after removed returns, 752
INFO: col set after removed last 5 games, 600
INFO: col set after removed Second Quarter Pts, 584
INFO: col set after removed Third Quarter, 568
INFO: col set after removed player offense, 388
INFO: Length of df train (2880, 388)
INFO: Length of df validate (720, 388)
INFO: Length of df test (721, 388)


In [52]:
#
# Replace NaN Values with Zero
#
df_train_o, df_validate_o, df_test_o = replace_any_nan_values(df_train_o, df_validate_o, df_test_o)

Found a difference for validate Away Avg. First Quarter Pts Scored last 1 games, 720 vs 719
Found a difference for validate Away Avg. First Quarter Pts Scored last 2 games, 720 vs 719
Found a difference for validate Away Avg. First Quarter Pts Scored last 3 games, 720 vs 719
Found a difference for validate Away Avg. Fourth Quarter Pts Scored last 1 games, 720 vs 719
Found a difference for validate Away Avg. Fourth Quarter Pts Scored last 2 games, 720 vs 719
Found a difference for validate Away Avg. Fourth Quarter Pts Scored last 3 games, 720 vs 719
Found a difference for validate Away Avg. Final Score last 1 games, 720 vs 719
Found a difference for validate Away Avg. Final Score last 2 games, 720 vs 719
Found a difference for validate Away Avg. Final Score last 3 games, 720 vs 719
Found a difference for validate Away Avg. First Quarter Pts Gave Up last 1 games, 720 vs 719
Found a difference for validate Away Avg. First Quarter Pts Gave Up last 2 games, 720 vs 719
Found a difference for

In [59]:
#
# Get the features-label sets for train, validate and test
#
x_train, y_train, x_validate, y_validate, x_test, y_test = get_feature_labels_train_validate_test_sets(df_train_o, df_validate_o, df_test_o)

INFO: The shape of the training feature set is (2880, 383)
INFO: The shape of the training labels is (2880,)
INFO: The shape of the validate feature set is (720, 383)
INFO: The shape of the validate labels is (720,)
INFO: The shape of the test feature set is (721, 383)
INFO: The shape of the test labels is (721,)


In [81]:
#
# Normalize the feature sets
#
x_train_norm, x_validate_norm, x_test_norm = normalize_test_validate_train_sets(x_train, x_test, x_validate)

In [82]:
#
# Run SVM for different kernels and parameters
#
ikernel_type = 'rbf'
clf_rbf = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_rbf.score(x_train_norm, y_train)}')

Validate Set Accuracy: 0.6125
Test Set Accuracy: 0.5922330097087378
INFO: The score for the SVM rbf classifier on the training set is 0.8774305555555556


In [83]:
ikernel_type = 'linear'
clf_linear = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_linear.score(x_train_norm, y_train)}')

Validate Set Accuracy: 0.5777777777777777
Test Set Accuracy: 0.5963938973647711
INFO: The score for the SVM linear classifier on the training set is 0.7243055555555555


In [87]:
ikernel_type = 'poly'
idegree_in = 3
clf_linear = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0, degree_in=idegree_in)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_linear.score(x_train_norm, y_train)}')
print(f'INFO: The polnomial degree is {idegree_in}')

Validate Set Accuracy: 0.5611111111111111
Test Set Accuracy: 0.5714285714285714
INFO: The score for the SVM poly classifier on the training set is 0.8958333333333334
INFO: The polnomial degree is 3


In [84]:
ikernel_type = 'poly'
idegree_in = 5
clf_linear = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0, degree_in=idegree_in)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_linear.score(x_train_norm, y_train)}')
print(f'INFO: The polnomial degree is {idegree_in}')

Validate Set Accuracy: 0.5694444444444444
Test Set Accuracy: 0.5672676837725381
INFO: The score for the SVM poly classifier on the training set is 0.8232638888888889
INFO: The polnomial degree is 5


In [85]:
ikernel_type = 'poly'
idegree_in = 10
clf_linear = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0, degree_in=idegree_in)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_linear.score(x_train_norm, y_train)}')
print(f'INFO: The polnomial degree is {idegree_in}')

Validate Set Accuracy: 0.5652777777777778
Test Set Accuracy: 0.5631067961165048
INFO: The score for the SVM poly classifier on the training set is 0.7666666666666667
INFO: The polnomial degree is 10


In [86]:
ikernel_type = 'poly'
idegree_in = 15
clf_linear = run_svm_nfl_predict(x_train_norm, y_train, 
                    x_validate_norm, y_validate, 
                    x_test_norm, y_test,
                    kernel_type=ikernel_type, C_in=1.0, degree_in=idegree_in)
print(f'INFO: The score for the SVM {ikernel_type} classifier on the training set is {clf_linear.score(x_train_norm, y_train)}')
print(f'INFO: The polnomial degree is {idegree_in}')

Validate Set Accuracy: 0.5652777777777778
Test Set Accuracy: 0.5672676837725381
INFO: The score for the SVM poly classifier on the training set is 0.7552083333333334
INFO: The polnomial degree is 15
